# M15 — Spectral & Temporal Representation Lab (Colab)

**Status of this notebook: Phase 1 (task T01) — authored, never executed by its author.** Sections for T02/T03/T07/T08 (signals, spectral features, encoders, preservation, latent, retrieval) are added in later sessions.

## A. Introduction

- **Objective.** Decide whether spectral (STFT / Welch / wavelet) and temporal (LSTM / GRU / TCN) representations of 10 Hz movement sequences are *reliable measurement instruments* — information-preserving and repeatable under comparable conditions — before any NFL prediction is attempted (that is M16–M18).
- **Hypotheses (H15.1–H15.5).** Spectral summaries carry athlete information that survives length matching; learned encoders preserve physical structure better than PCA; latent dimensions are repeatable; cross-domain retrieval beats position-only retrieval; STFT and LSTM are complementary. All are **[HYP]**.
- **Data.** The portable M12 bundle: 6 channels (`speed, a_tan, a_n, yaw, dt, heading_valid`, physical scale `[10, 5, 5, 3, 0.1, 1]`), ≤ 64 frames at 10 Hz, Combine + NFL sequences, 5 athlete-disjoint outer folds. No x/y, orientation, opponents or ball.
- **Dependencies.** Bundle only. The exploratory STFT/LSTM notebooks are read **only** through the provenance audit (section E); their numbers are [EXP].
- **This phase writes.** `tables/dataset/*`, `tables/legacy/prototype_audit`, `manifests/{fold_assignments.parquet, dataset_fingerprint.json, m15_status.json, m15_run_status.json}`, `datasets/cached/sequence_index`, figures `M15_A0_*` (PNG/SVG/PDF + data), `reports/markdown/m15_handoff.md`.
- **Gate.** G15 (`docs/research/M15_M18_SCIENTIFIC_GATES.md`) stays **NOT_EVALUATED** in this phase.

Every step ends in an explicit status: COMPLETED · CACHED · BLOCKED_MISSING_INPUT · BLOCKED_PROVENANCE · BLOCKED_INSUFFICIENT_SAMPLE · FAILED_VALIDATION · FAILED · NOT_RUN.

## B. Setup

In [ ]:
# Setup 1 — mount Drive and install nflbowl from GitHub (the Colab PyTorch / CUDA build is never replaced)
from google.colab import drive
drive.mount('/content/drive')
REPO = 'git+https://github.com/mybenkhadda/NFL_bowl.git@main'
%pip install -q {REPO}
%pip install -q --upgrade --force-reinstall --no-deps {REPO}
%pip install -q ipywidgets umap-learn PyWavelets   # optional: a missing package only blocks the steps that need it

In [ ]:
# Setup 2 — version check (after a reinstall: Runtime → Restart session, then continue from here)
import nflbowl
print('nflbowl', nflbowl.__version__)
assert tuple(int(x) for x in nflbowl.__version__.split('.')[:2]) >= (0, 7), 'nflbowl < 0.7.0 is imported: restart the session and rerun from Setup 2'
from pathlib import Path
import numpy as np
import polars as pl
from IPython.display import Markdown, display
from nflbowl.devices import resolve_device
from nflbowl.lab import handoff
from nflbowl.lab import status as S
from nflbowl.lab.m15 import config as m15cfg, data as m15data, legacy as m15legacy, optional as m15opt, plots as m15plots
display(m15opt.report())

In [ ]:
# Setup 3 — configuration (the ONLY cell to edit). Unknown keys raise; M12–M14 and the legacy folders are read only.
CFG = {
    'experiment_name': 'm15_spectral_temporal_v1',
    'bundle_dir': '/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1_clean',
    'drive_root': '/content/drive/MyDrive/NFLBowl',
    'legacy_m15_dirs': [],            # e.g. ['/content/drive/MyDrive/<exploratory M15 folder>'] — read only
    'legacy_split_manifests': [],     # optional explicit split files for those artifacts
    'device': 'auto',
    'seed': 42,
    'run_training': False, 'allow_new_training': False,   # Track B (T07) only; two opt-ins
}
cfg = m15cfg.resolve(CFG)
run = m15cfg.M15Run(cfg.run_dir, cfg)
env = handoff.environment(cfg.m13())
run.snapshot_config(cfg, env)
print('run dir           ', run.root)
print('config fingerprint', cfg.fingerprint())
print('device            ', resolve_device(cfg.device))
display(run.ledger.table())     # what earlier sessions already completed (resume)

## C. Data audit — E15.00 (M12 sequence-store adapter)

Opens and validates the bundle through `nflbowl.data.loading.open_bundle` (sizes, sha256 of small files, schemas), then `m15.data.audit_bundle`: fingerprint, channels, recorded scaling, shapes, row order, ids, domains, mask validity, physical plausibility on valid frames, heading convention, fold completeness / range / athlete-disjointness, linked athletes per fold, athletes-table contract. A failed contract check is **FAILED_VALIDATION** (tables are still written).

In [ ]:
# E15.00 — open, index, audit, persist (cached index reused when bundle + config fingerprints match)
def step_e15_00(ctx):
    data = m15data.prepare_m15_data(cfg, run, strict=False)
    ctx['outputs'] += ['tables/dataset/data_contract_audit', 'manifests/fold_assignments.parquet', 'manifests/dataset_fingerprint.json']
    m15data.assert_contract(data.audit)          # FAILED_VALIDATION after the evidence is saved
    return data

DATA = S.run_step(run, run.ledger, 'E15.00', step_e15_00, note='bundle audit + sequence index')
if DATA is None:
    display(run.ledger.table())
    raise SystemExit('E15.00 did not complete — read the reason above (and logs/e15_00_traceback.txt); later sections need the bundle.')
display(DATA.audit)
print('game key:', DATA.game_key_status)

In [ ]:
# Counts — sequences, athletes, trials per drill, linked cohort (re-read from the saved tables)
for name in ('valid_frames_by_domain', 'trials_per_drill', 'linked_cohort', 'counts_by_domain_kind_position'):
    display(Markdown(f'**{name}**')); display(run.frame(name, 'dataset'))
fa = DATA.folds
display(fa.group_by('fold', 'fold_source').agg(pl.len().alias('athletes'), (pl.col('has_combine') & pl.col('has_nfl')).sum().alias('both_domains')).sort('fold', 'fold_source'))

In [ ]:
# Figures — audit summary and valid-frame distributions (PNG / SVG / PDF + source data under figures/)
fig, dat = m15plots.audit_status_bar(DATA.audit)
run.figure(fig, 'M15_A0_01_audit_status', data=dat, question='Does the bundle satisfy the M15 data contract?', caption='Data-contract checks per status.')
fig, dat = m15plots.valid_frames_hist(DATA.index, cfg.max_frames)
run.figure(fig, 'M15_A0_02_valid_frames', data=dat, question='How long are Combine vs NFL sequences?',
           caption='Share of sequences by number of valid frames, per domain (Finding 2: NFL windows are shorter).')

## D. Exploratory analysis — sequence explorer

Physical-unit traces; invalid frames are gaps (never zeros); yaw is drawn only where heading is valid. Select athlete, domain and drill / demand kind; tick *save* to write the current view with its data.

In [ ]:
import matplotlib.pyplot as plt
idx = DATA.index
def show_sequences(nfl_id, domain, kind, max_n=6, save=False):
    sel = idx.filter((pl.col('nfl_id') == nfl_id) & (pl.col('domain') == domain) & ((pl.col('kind') == kind) if kind != '(all)' else pl.lit(True))).head(max_n)
    if not sel.height:
        print('no sequence for this selection'); return
    labels = [f"{r['kind']} · {r['seq_key']} · fold {r['fold']}" for r in sel.iter_rows(named=True)]
    fig, dat = m15plots.signal_panel(DATA.bundle, sel['idx'].to_numpy(), labels)
    if save:
        run.figure(fig, f'M15_A0_03_sequences_{nfl_id}_{domain}', data=dat, question='What do the raw movement signals look like?', caption='; '.join(labels))
    else:
        plt.show()

if m15opt.available('ipywidgets'):
    import ipywidgets as W
    ath = sorted(idx['nfl_id'].unique().to_list())
    w_ath = W.Dropdown(options=ath, description='athlete')
    w_dom = W.ToggleButtons(options=['combine', 'nfl'], description='domain')
    w_kind = W.Dropdown(options=['(all)'] + sorted(idx['kind'].unique().to_list()), description='kind')
    W.interact(show_sequences, nfl_id=w_ath, domain=w_dom, kind=w_kind, max_n=W.IntSlider(value=6, min=1, max=12), save=W.Checkbox(value=False))
else:
    first = int(idx.filter(pl.col('domain') == 'combine')['nfl_id'][0])
    show_sequences(first, 'combine', '(all)')

## E. Experiments — E15.00-L: legacy (exploratory) STFT / LSTM artifact provenance

Classifies every file under `legacy_m15_dirs` as **verified_fold_safe**, **verified_exploratory_only** or **unknown_provenance** (rules in `nflbowl/lab/m15/legacy.py`). Sources are never modified. Only `verified_fold_safe` artifacts may enter supervised out-of-fold evaluation (`legacy.assert_fold_safe`). With no folder configured the step is **BLOCKED_MISSING_INPUT** — expected until the exploratory paths are supplied.

In [ ]:
def step_legacy(ctx):
    if not cfg.legacy_m15_dirs:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'no legacy_m15_dirs configured (Setup 3)', missing=['legacy_m15_dirs'])
    missing = [d for d in cfg.legacy_m15_dirs if not Path(d).exists()]
    if missing:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'legacy folder(s) not found on Drive', missing=missing)
    tab, recs = m15legacy.audit(cfg.legacy_m15_dirs, DATA.bundle, cfg.outer_folds, cfg.legacy_split_manifests, DATA.fold_of)
    ctx['outputs'] += run.table(tab, 'prototype_audit', 'legacy') + run.table(m15legacy.summary(recs), 'prototype_audit_summary', 'legacy')
    return tab, recs

LEGACY = S.run_step(run, run.ledger, 'E15.00-L', step_legacy, note='legacy STFT / LSTM provenance audit', requires=['E15.00'])
if LEGACY is not None:
    display(run.frame('prototype_audit_summary', 'legacy'))
    display(LEGACY[0].select('path', 'kind', 'provenance', 'reasons', 'n_rows', 'n_athletes', 'latent_dim'))
else:
    display(run.ledger.table().filter(pl.col('step') == 'E15.00-L'))

## F. Results, gate status and next experiments

Phase 1 computes no representation metric, so **G15 is NOT_EVALUATED** by construction. Next tasks: T02 (E15.A1–A2 signal analytics), T03 (E15.A3–A4 spectral resolution-artifact test), T07–T08 (encoders, preservation, latent, retrieval).

In [ ]:
gate = pl.DataFrame([S.gate_row(c, None, None, None, note='no representation evaluated in Phase 1') for c in
                     ('G15.a preservation', 'G15.b event preservation', 'G15.c repeatability', 'G15.d no collapse', 'G15.e seed stability')])
run.table(gate, 'm15_gate', 'synthesis')
display(gate)
display(run.ledger.table())

## G. Persistence and hand-off

In [ ]:
run.write_status()
run.save_markdown(run.handoff_markdown(), 'm15_handoff')
for rel in ('manifests/m15_status.json', 'manifests/m15_run_status.json', 'manifests/dataset_fingerprint.json', 'manifests/fold_assignments.parquet',
            'tables/dataset', 'tables/legacy', 'figures/png', 'reports/markdown'):
    p = run.root / rel
    print(('✓ ' if p.exists() else '· ') + str(p))
print('\nSEND BACK: manifests/stages.json, manifests/m15_status.json, logs/*_traceback.txt, tables/dataset/data_contract_audit.csv,')
print('           tables/dataset/trials_per_drill.csv, tables/dataset/linked_cohort.csv, tables/legacy/prototype_audit.csv (if run)')